# Lab 1: Predicting the Price of Opportunity
## How Governments Use Regression to Shape Economic Policy

**Duration:** 45 minutes | **Theme:** Tell a story through data

---

### The Story

A government wants to design better housing subsidies and financial inclusion policies. Their challenge: **What drives credit demand?**

If they can predict how much credit people need based on their profile (age, employment, savings), they can:
- Design targeted housing subsidies
- Set fair tax brackets
- Identify underserved populations
- Shape economic policy with data

**Your role:** You are the data scientist advising the government.

In Lab 0, you cleaned and explored this data. Now, let's make it **PREDICT**.

---

### Remember: You're not just building a model. You're telling a story about society.

## Setup: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load the German Credit dataset
url = 'https://raw.githubusercontent.com/sampathlonka/ml-workshop/master/Data/german_credit_data.csv'

# Load data (CSV with headers)
df = pd.read_csv(url)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Quick data quality check
print("Missing values:")
print(df.isnull().sum())
print(f"\nData types:\n{df.dtypes}")

---

# Act 1: What is Regression? (The Economist's Lens)

**Regression answers one simple question: "How much?"**

Real-world analogy:
> If a person is 30 years old, has moderate savings, works in a stable job, and wants to buy a home — how much credit will they need?

Regression finds the pattern in data to answer this question for thousands of people.

### Your Question for Discussion

**What factors do YOU think drive the amount of credit someone needs?**

Think about:
- Age (young vs. established professionals)
- Job type (stable vs. freelance)
- Savings (having a safety net)
- Loan duration they want to pay back over
- Existing debts

*Discuss with your neighbor for 1 minute.*

### Let's Visualize the Relationship

We'll start with a simple question: **Does Age affect Loan Amount?**

In [ ]:
# Simple scatter plot: Age vs Loan Amount
plt.figure(figsize=(10, 6))
plt.scatter(df['Age'], df['LoanAmount'], alpha=0.5, s=30, color='steelblue')

# Add a simple trend line
z = np.polyfit(df['Age'], df['LoanAmount'], 1)
p = np.poly1d(z)
plt.plot(df['Age'].sort_values(), p(df['Age'].sort_values()), "r--", linewidth=2, label='Trend')

plt.xlabel('Age (years)', fontsize=12)
plt.ylabel('Loan Amount (in currency units)', fontsize=12)
plt.title('Does Age Drive Credit Demand?', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate correlation
corr = df['Age'].corr(df['LoanAmount'])
print(f"Correlation between Age and Loan Amount: {corr:.3f}")
print(f"\nInterpretation: {['Weak', 'Moderate', 'Strong'][min(2, int(abs(corr)*3))]} relationship")

### INSIGHT 💡

One feature alone doesn't tell the whole story. People are complex. Let's add more features to understand the full picture.

---

# Act 2: Building the Model

### Step 1: Prepare the Data

We'll select features that we think drive credit demand. **But first, a policy question:**

**POLICY INSIGHT 🏛️:** If we included Gender as a feature, what would happen?
- The model might predict higher credit for men, lower for women (if historical bias exists in the data)
- This could perpetuate discrimination in lending
- Even if "statistically useful," should we use it?

**Answer:** For this lab, we avoid gender. We choose features that are ethically defensible.

In [ ]:
# Select features based on economic reasoning
# LoanDuration: How long they want to pay back (affects total amount)
# Age: Life stage and earning capacity
# Job: Employment stability affects credit worthiness
# ExistingSavings: Financial cushion

# Prepare data: we need numeric features
X = df[['LoanDuration', 'Age']].copy()  # Start simple: 2 features
y = df['LoanAmount'].copy()  # Target: Loan Amount

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature statistics:")
print(X.describe())

In [ ]:
# Split data into training (80%) and testing (20%) sets
# This allows us to evaluate on unseen data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
# Train the regression model
model = LinearRegression()
model.fit(X_train, y_train)

print("✓ Model trained successfully!")

### Step 2: Interpret the Coefficients

**Every coefficient tells a story about society.**

In [ ]:
# Extract and display coefficients
coefficients = pd.DataFrame({
    'Feature': ['Loan Duration', 'Age'],
    'Coefficient': model.coef_
})

print("Regression Coefficients:")
print(coefficients)
print(f"\nIntercept (baseline loan amount): {model.intercept_:.2f}")

print("\n" + "="*60)
print("INTERPRETATION:")
print("="*60)
for idx, row in coefficients.iterrows():
    feature = row['Feature']
    coef = row['Coefficient']
    print(f"\n{feature}:")
    if coef > 0:
        print(f"  For every 1-unit increase in {feature}, loan amount increases by {coef:.2f}")
    else:
        print(f"  For every 1-unit increase in {feature}, loan amount decreases by {abs(coef):.2f}")

### POLICY INSIGHT 🏛️

Look at the Loan Duration coefficient. It's likely **positive and large**.

**What does this mean?**
- People who want longer repayment periods borrow more
- Makes sense: home loans are larger than car loans
- **Policy implication:** For affordable housing policy, we should allow longer repayment periods for low-income borrowers

Now look at Age. What does that coefficient tell us?
- If positive: older people borrow more (they have higher lifetime earnings)
- If negative: younger people borrow more (urgent life needs)
- **Policy implication:** Should young borrowers get preferential rates?

---

# Act 3: Visualizing the Predictions

Now we see how our model performs on the test set.

In [ ]:
# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print(f"Sample predictions (first 5 test samples):")
print(f"\nActual vs Predicted:")
comparison = pd.DataFrame({
    'Actual': y_test.head(5).values,
    'Predicted': y_pred_test[:5]
})
print(comparison)

In [ ]:
# Visualization 1: Actual vs Predicted
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_test, alpha=0.6, s=40, color='steelblue', label='Predictions')

# Perfect prediction line
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

plt.xlabel('Actual Loan Amount', fontsize=12)
plt.ylabel('Predicted Loan Amount', fontsize=12)
plt.title('Actual vs Predicted: Where Does Our Model Succeed?', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Residuals (Prediction Errors)
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual scatter plot
axes[0].scatter(y_pred_test, residuals, alpha=0.6, s=40, color='coral')
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Loan Amount', fontsize=11)
axes[0].set_ylabel('Residuals (Actual - Predicted)', fontsize=11)
axes[0].set_title('Residual Plot: Where Are We Off?', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=30, color='teal', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Residual Value', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Distribution of Errors', fontsize=12, fontweight='bold')
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.2f} (should be close to 0)")
print(f"Std dev of residuals: {residuals.std():.2f}")

In [ ]:
# Visualization 3: Loan Duration vs Loan Amount with prediction line
# For visualization, hold Age constant at its mean
age_mean = X_test['Age'].mean()

# Create range of Loan Duration values
duration_range = np.linspace(X['LoanDuration'].min(), X['LoanDuration'].max(), 100)
X_pred = pd.DataFrame({
    'LoanDuration': duration_range,
    'Age': age_mean
})
y_pred_range = model.predict(X_pred)

plt.figure(figsize=(10, 6))
plt.scatter(X_test['LoanDuration'], y_test, alpha=0.5, s=40, color='steelblue', label='Actual data')
plt.plot(duration_range, y_pred_range, 'r-', linewidth=3, label='Model prediction')

plt.xlabel('Loan Duration (months)', fontsize=12)
plt.ylabel('Loan Amount', fontsize=12)
plt.title(f'Loan Duration vs Loan Amount (Age fixed at {age_mean:.0f} years)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### INSIGHT 💡

**Look at the residual plot. The points that are far from zero are where our model fails.**

Those failures tell us about people the system doesn't understand:
- Large positive residuals: People who borrowed WAY more than we predicted (risk-takers? entrepreneurs?)
- Large negative residuals: People who borrowed LESS than we predicted (conservative? scared?)

### POLICY INSIGHT 🏛️

**If the model systematically under-predicts credit needs for young borrowers, it means:**
- Current lending formulas may underestimate youth credit demand
- Young people might face policy barriers
- Government could design targeted interventions

**Data reveals not just what IS, but what SHOULD BE.**

---

# Act 4: How Good is Our Crystal Ball?

### Evaluating Model Performance

We use three metrics, explained in plain English (no math jargon required).

In [ ]:
# Calculate performance metrics
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("MODEL PERFORMANCE REPORT")
print("="*60)
print(f"\nR² Score (Test Set): {r2_test:.3f}")
print(f"  → Our model explains {r2_test*100:.1f}% of loan variation")
print(f"  → The remaining {(1-r2_test)*100:.1f}% is unexplained (human complexity!)")

print(f"\nMean Absolute Error (MAE): {mae:.2f}")
print(f"  → On average, predictions are off by {mae:.2f} currency units")
print(f"  → Actual mean loan amount: {y_test.mean():.2f}")
print(f"  → Error as % of mean: {(mae/y_test.mean())*100:.1f}%")

print(f"\nRoot Mean Squared Error (RMSE): {rmse:.2f}")
print(f"  → Larger errors penalized more heavily")
print(f"  → Useful for detecting outlier mistakes")

print(f"\nOverfitting Check:")
print(f"  → R² on training data: {r2_train:.3f}")
print(f"  → R² on test data: {r2_test:.3f}")
if abs(r2_train - r2_test) < 0.05:
    print(f"  → Good: Model generalizes well (no overfitting)")
else:
    print(f"  → Model may be overfitting or underfitting")

### INTERPRETATION: What Does R² = 0.6 (or whatever it is) Mean?

**Plain English Version:**

Imagine you're trying to explain to someone why people borrow different amounts. You say:
- "Loan Duration and Age matter!"
- Your friend says, "But what about their education, existing debt, family size, local economy?"

Your friend is right. **R² tells us: you've explained 60% of the story, but 40% is missing.**

That 40% is real. It's people's hopes, fears, surprises, and life circumstances we can't capture in simple numbers.

In [ ]:
# Comparison: Simple model vs Complex model
# Simple: only Loan Duration
X_simple = df[['LoanDuration']]
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_simple, y, test_size=0.2, random_state=42)
model_simple = LinearRegression().fit(X_train_s, y_train_s)
r2_simple = r2_score(y_test_s, model_simple.predict(X_test_s))

# Complex: Loan Duration + Age (our current model)
r2_current = r2_test

comparison_df = pd.DataFrame({
    'Model': ['Simple (Loan Duration only)', 'Current (Loan Duration + Age)'],
    'R² Score': [r2_simple, r2_current],
    'Improvement': ['-', f'+{(r2_current - r2_simple)*100:.1f}%']
})

print("\nMODEL COMPARISON")
print(comparison_df.to_string(index=False))
print(f"\nAdding Age as a feature improved our model by {(r2_current - r2_simple)*100:.1f} percentage points.")

### DISCUSSION: When Is "Good Enough" Good Enough? 🤔

**A model with R² = 0.6 is useful for policy IF:**

✓ It identifies the general trends (loan needs DO increase with duration)  
✓ It helps understand which groups are underserved  
✓ It points toward better policies (not worse ones)  

**It's NOT good enough IF:**

✗ You use it to deny individual loans (too many false negatives)  
✗ You assume it explains causation (correlation ≠ causation)  
✗ You ignore the 40% of unexplained variation as "not important"  

### KEY INSIGHT: Correlation vs Causation 🔍

Our model shows: **Loan Duration is strongly correlated with Loan Amount**

But does longer duration CAUSE higher credit?
- Interpretation A (causal): "If we encourage longer repayment, people will borrow more"
- Interpretation B (correlation): "People who need more money choose longer repayment periods to make payments affordable"

**Which is correct?** Usually B. The causality runs the OTHER way.

This is why economists build causal models, not just regression. It's also why policy is hard.

---

# Act 5: Wrap-Up & Bridge to Tomorrow

## What We Did Today

In [ ]:
# Summary statistics
print("📈 LAB 1 SUMMARY")
print("="*60)
print(f"\n1. We asked: How much credit do people need?")
print(f"\n2. We used features: Loan Duration, Age")
print(f"\n3. We built a linear regression model")
print(f"\n4. Model performance: R² = {r2_test:.3f}")
print(f"   → Explains {r2_test*100:.1f}% of loan variation")
print(f"   → Average error: {mae:.2f} currency units")
print(f"\n5. Key findings:")
print(f"   → Loan Duration is a strong predictor (coef = {model.coef_[0]:.2f})")
print(f"   → Age matters but less than duration (coef = {model.coef_[1]:.2f})")
print(f"   → Many people don't fit the pattern (residuals!)")
print(f"\n" + "="*60)

## Three Policy Recommendations from Today's Analysis

### 1. **Duration-Based Subsidy Design**
Since duration strongly predicts loan amount, design housing subsidies that vary by repayment timeline:
- Long-term mortgages (25+ years) for affordable housing
- Flexible terms to accommodate life stages

### 2. **Age-Aware Financial Inclusion**
Our model shows different age groups have different credit needs:
- Target youth with education and first-time homebuyer programs
- Recognize that older borrowers may have different profiles

### 3. **Monitor Model Failures**
The people our model doesn't predict well are telling us something:
- Those with large positive residuals: entrepreneurs needing growth capital
- Those with large negative residuals: risk-averse savers
- Design programs for these under-understood groups

## Tomorrow: Lab 2 - Classification

Today we predicted **HOW MUCH** credit someone needs (regression).  
Tomorrow we'll predict **YES or NO** - whether to approve a credit application (classification).

**The connection:**
- Regression tells us magnitude: "This person needs 5,000 units of credit"
- Classification tells us category: "Approve or deny?"
- Together, they shape lending policy

**Remember:** Every coefficient in your model tells a story about society. Use that power wisely.

---

## Optional: Explore Further

If you have extra time, try these extensions:
1. **Add more features:** Include EmploymentDuration, ExistingSavings, etc. Does R² improve?
2. **Non-linear relationships:** Does the relationship look curved, not straight?
3. **Subgroup analysis:** Do young vs. old borrowers follow different patterns?
4. **Feature engineering:** What if you created new features (e.g., Age × LoanDuration)?

In [ ]:
# BONUS: Quick feature exploration
print("Correlation of numeric features with Loan Amount:")
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlations = df[numeric_cols].corr()['LoanAmount'].sort_values(ascending=False)
print(correlations)
print("\nWhich features are most correlated with loan amount?")
print("Consider adding them to your model!")

---

**End of Lab 1**

*"You predicted how much. Tomorrow you'll decide yes or no. Together, data shapes policy."* 🏛️